# Matching Lab — Let the Math Find Your Twin

Your class survey becomes **data**. Each person is a row of 1-5 answers (a vector), and we measure how alike any two people are with **cosine similarity** — the same math from this morning.

**What you will do**
1. Upload your class CSV
2. Check one similarity by hand
3. Radar profile of one person
4. Compatibility checker (colour-coded)
5. Heatmap + "most similar" top-5
6. K-means suggests teams
7. Score each team's balance

> Run the cells one at a time, in order. Before each one, PREDICT what will appear. No API keys, no GPU.

## 1. Setup

In [ ]:
!pip install -q seaborn scikit-learn ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
print("Ready")

## 2. Upload your class CSV

Export your Google Form responses (Responses tab -> three-dot menu -> Download responses (.csv)) and upload here. Columns are auto-detected, so any number of questions works. Your form just needs a short-answer **Nickname** question plus the 1-5 questions.

In [ ]:
from google.colab import files

print("Choose your class_survey.csv ...")
uploaded = files.upload()

csv_files = [f for f in uploaded.keys() if f.endswith(".csv")]
if not csv_files:
    raise SystemExit("No .csv found - run this cell again.")

data = pd.read_csv(csv_files[0])

def looks_like_timestamp(name):
    n = str(name).lower()
    return ("time" in n) or ("date" in n)

def looks_like_nickname(name):
    n = str(name).lower()
    return any(k in n for k in ["nick", "name", "handle"])

cols = [c for c in data.columns if not looks_like_timestamp(c)]
name_cols = [c for c in cols if looks_like_nickname(c)]
if name_cols:
    nickname_col = name_cols[0]
else:
    nickname_col = cols[0]
    print("NOTE: no 'Nickname' question found - using:", nickname_col)
    print("Add a short-answer 'Nickname' question to your form for clean labels.")

question_cols = [c for c in cols
                 if c != nickname_col and pd.to_numeric(data[c], errors="coerce").notna().all()]

nicknames = data[nickname_col].astype(str).values
student_vectors = data[question_cols].astype(float).values

print(f"People: {len(nicknames)}")
print(f"Questions: {len(question_cols)}")
print(f"Name column: {nickname_col}")
display(data[[nickname_col] + list(question_cols[:4])].head())

## 3. Cosine similarity — and a by-hand check

Build an N x N table of how alike every pair is, then re-compute ONE pair by hand to prove the library uses the morning's exact formula.

In [ ]:
sim = cosine_similarity(student_vectors)
sim_df = pd.DataFrame(sim, index=nicknames, columns=nicknames)
display(sim_df.iloc[:5, :5].round(3))

In [ ]:
if len(nicknames) >= 2:
    a, b = student_vectors[0], student_vectors[1]
    dot = float(np.dot(a, b))
    len_a, len_b = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    by_hand = dot / (len_a * len_b)
    by_lib = float(sim_df.iloc[0, 1])
    print(f"{nicknames[0]} x {nicknames[1]}")
    print(f"  dot (a . b)     = {dot:.4f}")
    print(f"  |a|, |b|        = {len_a:.4f}, {len_b:.4f}")
    print(f"  by hand         = {by_hand:.6f}")
    print(f"  by scikit-learn = {by_lib:.6f}")
    print("  match? =", "YES" if np.isclose(by_hand, by_lib) else "NO")

## 4. Radar profile

Pick a classmate and see their answers as a shape. Two people with a similar shape have a high cosine similarity.

In [ ]:
def launch_personality_quiz():
    selector = widgets.Dropdown(options=list(nicknames), description="Person", layout=widgets.Layout(width="320px"))
    show_btn = widgets.Button(description="Show radar", button_style="success", layout=widgets.Layout(width="180px", height="40px"))
    out = widgets.Output()

    def show_radar(_=None):
        with out:
            clear_output()
            name = selector.value
            row = data[data[nickname_col].astype(str) == str(name)]
            if row.empty:
                print("Not found:", name)
                return
            values = row[question_cols].astype(float).values.flatten().tolist()
            labels = [str(c)[:14] for c in question_cols]
            angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
            angles += angles[:1]
            plot_vals = values + [values[0]]
            fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
            ax.plot(angles, plot_vals, color="#7C3AED", linewidth=2)
            ax.fill(angles, plot_vals, alpha=0.25, color="#7C3AED")
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(labels, fontsize=9)
            ax.set_ylim(0, 5)
            ax.set_title(f"{name} - personality profile", fontsize=15, pad=20)
            plt.tight_layout()
            plt.show()
            hi, lo = int(np.argmax(values)), int(np.argmin(values))
            display(HTML(
                f"<div style='background:#f3f0ff;padding:16px;border-radius:10px;'>"
                f"<b>Average:</b> {np.mean(values):.2f}/5.0 &nbsp; "
                f"<b>Highest:</b> {question_cols[hi]} ({values[hi]:.0f}) &nbsp; "
                f"<b>Lowest:</b> {question_cols[lo]} ({values[lo]:.0f})</div>"))

    show_btn.on_click(show_radar)
    display(widgets.HTML("<p>Pick a person, then press the button.</p>"))
    display(selector, show_btn, out)

launch_personality_quiz()

## 5. Compatibility checker

Pick two people. The score is their cosine similarity, colour-coded: green >= 0.90 (very alike), blue >= 0.70 (alike), orange >= 0.50 (so-so), red < 0.50 (quite different). Check at least three pairs, predicting each first.

In [ ]:
def launch_compatibility_checker():
    p1 = widgets.Dropdown(options=list(nicknames), description="Person 1", layout=widgets.Layout(width="240px"))
    p2 = widgets.Dropdown(options=list(nicknames), description="Person 2", layout=widgets.Layout(width="240px"))
    btn = widgets.Button(description="Check", button_style="primary", layout=widgets.Layout(width="160px", height="40px"))
    out = widgets.Output()

    def check(_=None):
        with out:
            clear_output()
            n1, n2 = p1.value, p2.value
            if n1 == n2:
                display(HTML("<p style='color:#ef4444;'>Pick two different people.</p>"))
                return
            score = float(sim_df.loc[n1, n2])
            if score >= 0.90:
                color, msg = "#10b981", "very alike"
            elif score >= 0.70:
                color, msg = "#3b82f6", "alike"
            elif score >= 0.50:
                color, msg = "#f59e0b", "so-so"
            else:
                color, msg = "#ef4444", "quite different"
            display(HTML(
                f"<div style='background:{color};padding:22px;border-radius:12px;color:white;text-align:center;'>"
                f"<h3 style='margin:0;'>{n1} x {n2}</h3>"
                f"<p style='font-size:34px;margin:8px 0;'>{score:.3f}</p>"
                f"<p style='font-size:18px;margin:0;'>{msg}</p></div>"
                f"<div style='background:#f8fafc;padding:12px;border-radius:10px;margin-top:8px;'>"
                f"A big difference is not bad - it is a chance to complement each other.</div>"))

    btn.on_click(check)
    display(widgets.HTML("<p>Pick two people and check.</p>"))
    display(widgets.HBox([p1, p2]), btn, out)

launch_compatibility_checker()

## 6. Whole-class dashboard

The heatmap shows every pair at once (darker = more alike). "Find similar" ranks the top-5 for whoever you choose.

In [ ]:
def launch_visualization_dashboard():
    base = widgets.Dropdown(options=list(nicknames), description="Person", layout=widgets.Layout(width="260px"))
    heat_btn = widgets.Button(description="Heatmap", button_style="info", layout=widgets.Layout(width="160px", height="40px"))
    top_btn = widgets.Button(description="Find similar", button_style="success", layout=widgets.Layout(width="160px", height="40px"))
    out = widgets.Output()

    def show_heatmap(_):
        with out:
            clear_output()
            size = max(8, min(16, len(nicknames) * 0.6))
            plt.figure(figsize=(size, size * 0.85))
            sns.heatmap(sim_df, cmap="YlOrRd", vmin=0, vmax=1, annot=len(nicknames) <= 16, fmt=".2f", annot_kws={"size": 8})
            plt.title("Cosine similarity heatmap", fontsize=14)
            plt.tight_layout()
            plt.show()

    def show_top(_):
        with out:
            clear_output()
            b = base.value
            ranked = sim_df.loc[b].drop(b).sort_values(ascending=False).head(5)
            rows = "".join(
                f"<tr><td style='padding:8px;'>{i}</td><td style='padding:8px;'>{n}</td>"
                f"<td style='padding:8px;'><b>{s:.3f}</b></td></tr>"
                for i, (n, s) in enumerate(ranked.items(), 1))
            display(HTML(
                f"<div style='background:#eef2ff;padding:18px;border-radius:12px;'>"
                f"<h3 style='margin-top:0;'>Most similar to {b}</h3>"
                f"<table style='width:100%;border-collapse:collapse;'>"
                f"<thead style='background:#c7d2fe;'><tr><th style='padding:8px;text-align:left;'>#</th>"
                f"<th style='padding:8px;text-align:left;'>Name</th>"
                f"<th style='padding:8px;text-align:left;'>Score</th></tr></thead>"
                f"<tbody>{rows}</tbody></table></div>"))

    heat_btn.on_click(show_heatmap)
    top_btn.on_click(show_top)
    display(widgets.HTML("<p>See the whole class, or find one person's closest matches.</p>"))
    display(base, widgets.HBox([heat_btn, top_btn]), out)

launch_visualization_dashboard()

## 7. K-means suggests teams

Nobody labels anyone. K-means drops a few centers, pulls each person to the nearest, moves the centers, and repeats until they settle. PCA flattens the answers to 2D so we can see the islands. Change `n_teams` and re-run.

In [ ]:
n_teams = 4  # @param {type:"slider", min:2, max:6, step:1}

km = KMeans(n_clusters=n_teams, random_state=42, n_init=10)
labels = km.fit_predict(student_vectors)
coords = PCA(n_components=2).fit_transform(student_vectors)

plt.figure(figsize=(11, 8))
palette = plt.cm.Set2(np.linspace(0, 1, n_teams))
for t in range(n_teams):
    m = labels == t
    plt.scatter(coords[m, 0], coords[m, 1], color=[palette[t]], s=120, alpha=0.75, label=f"Team {t + 1}")
    for x, y, name in zip(coords[m, 0], coords[m, 1], nicknames[m]):
        plt.annotate(str(name), (x, y), xytext=(5, 5), textcoords="offset points", fontsize=9)
plt.xlabel("PCA axis 1")
plt.ylabel("PCA axis 2")
plt.title("Suggested teams (K-means, shown in 2D)", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. How many teams? The elbow

Run K-means for k = 1..6 and plot how tightly people sit inside their team. The spread always shrinks, but where the line suddenly bends — the elbow — is the honest number of teams.

In [ ]:
ks = range(1, min(7, len(nicknames)))
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(student_vectors).inertia_ for k in ks]
plt.figure(figsize=(8, 5))
plt.plot(list(ks), inertias, "o-", color="#7C3AED", linewidth=2)
plt.xlabel("number of teams (k)")
plt.ylabel("spread inside teams")
plt.title("Elbow method")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Look for the bend.")

## 9. Score each team's balance

Two scores per team: **diversity** (1 - average similarity inside the team; higher = more varied views) and **harmony** (the lowest similarity in the team; higher = nobody clashes hard). A good team is not just "all alike".

In [ ]:
def team_scores(idx):
    if len(idx) < 2:
        return float("nan"), float("nan")
    pairs = [sim[i, j] for i, j in combinations(idx, 2)]
    return 1 - float(np.mean(pairs)), float(np.min(pairs))

for t in range(n_teams):
    idx = np.where(labels == t)[0]
    members = ", ".join(map(str, nicknames[idx]))
    div, harm = team_scores(idx)
    print(f"Team {t + 1} ({len(idx)}): {members}")
    if not np.isnan(div):
        print(f"   diversity = {div:.3f}   harmony = {harm:.3f}")
    print()

## 10. Break the seal

Unfold your morning sticky note. Did the math put your guessed twin in your cluster, or near you on the heatmap? Tally the whole class: how often did **gut** match **math**?

- A cluster is **not** a verdict on you — it reflects a few answers on one afternoon.
- Being an outlier means you bring a rare angle: individuality, not an error.
- Teams here are a **hint** for later, not a final decision.